## LangGraph: Intelligent Graphs

1. Stateful graphs with memory
2. Conditional routing & decisions
3. Loops & iterative refinement
4. Human-in-the-loop workflows
5. Parallel execution
6. Production-ready agents

In [1]:
from langchain_ollama import ChatOllama
   
llm = ChatOllama(
    model='llama3.1',
    # base_url=os.getenv("OPENAI_API_BASE"),
    # api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0.3,
    max_tokens=50
)

In [2]:
#!/usr/bin/env python3
"""
Task 2: Stateful Graph - Simple Demo
Shows that LangGraph maintains state across steps
"""

from typing import TypedDict
from langgraph.graph import StateGraph, END

# Define state structure
class ConversationState(TypedDict):
    name: str
    greeting: str
    farewell: str

print("\n🕸️ STATEFUL GRAPH DEMO")
print("=" * 40)

# Input
name = "Alice"
print(f"Input: {name}\n")

# Define nodes that use state
def greet_person(state: ConversationState):
    """Step 1: Greet and save name to state"""
    prompt = f"Say hello to {state['name']} in 5 words or less"
    greeting = llm.invoke(prompt).content
    print(f"Step 1: {greeting} (saved name to state)")
    return {"greeting": greeting}

def say_farewell(state: ConversationState):
    """Step 2: Use saved name from state"""
    prompt = f"Say goodbye to {state['name']} mentioning their name in 5 words or less"
    farewell = llm.invoke(prompt).content
    print(f"Step 2: {farewell} (used name from state)")
    return {"farewell": farewell}

def check_memory(state: ConversationState):
    """Test: Can access saved state"""
    print(f"\nMemory Test: The person's name is {state['name']}")
    print("→ State preserved! Graph remembers everything.\n")
    return {}

# Build the graph
workflow = StateGraph(ConversationState)
workflow.add_node("greet", greet_person)
workflow.add_node("farewell", say_farewell)
workflow.add_node("memory", check_memory)

# Define flow
workflow.set_entry_point("greet")
workflow.add_edge("greet", "farewell")
workflow.add_edge("farewell", "memory")
workflow.add_edge("memory", END)

# Compile and run
app = workflow.compile()
result = app.invoke({"name": name})
result


🕸️ STATEFUL GRAPH DEMO
Input: Alice

Step 1: Hello, Alice. How are you? (saved name to state)
Step 2: "Goodbye, dear. I'll see you." (used name from state)

Memory Test: The person's name is Alice
→ State preserved! Graph remembers everything.



{'name': 'Alice',
 'greeting': 'Hello, Alice. How are you?',
 'farewell': '"Goodbye, dear. I\'ll see you."'}